In [ ]:
### My libraries (not pip)
try:
    import mathlib as Mathlib   #< c++ version (much faster training). Must be compiled first
    from NeuralNetwork_CPP import Batch, ActivationType
    print("Using C++ compiled mathlib!")
except ImportError:
    import PYTHON_Network.Mathlib as Mathlib
    from PYTHON_Network.ActivationTypes import ActivationType
    from PYTHON_Network.Batch import Batch
    print("Using python mathlib as the C++ compiled library is not avaialble!")

import PYTHON_Network.Loss as Loss
import PYTHON_Network.Optimizer as Optimizer
import PYTHON_Network.Accuracy as Accuracy


Using C++ compiled mathlib!


#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/SoftmaxCrossEntropyDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



In [5]:
import json
import time
start = time.time()
BATCH_SIZE = int(1000/20)#32

with open("datasets/squareData.json", "r") as f:
    a = json.load(f)
    X = a["X"]         #< 32x32 image 1 = pixel, 0 = no pixel
    y = a["y"]

print("Converting data to hilbert space")
for i in range(len(X)):
    X[i] = Mathlib.hilbertFlatten(X[i])   #< map to 1D using hilbert space (hilbert is used here so the resolution has minimal effect)
    print(f"{i}/{len(X)}", end="\r")
print(f"\rDone/{len(X)}")

spiralLayer1 = Batch(BATCH_SIZE, 32*32, 32, ActivationType.LEAKY_RELU)
spiralLayer2 = Batch(BATCH_SIZE, 32, 2, ActivationType.PASS)

try:
    with open("trainedModel.json", "r") as f:
        savedModel = json.load(f)
    spiralLayer1.setWeights(savedModel["layer1"]["weights"])
    spiralLayer1.setBiases(savedModel["layer1"]["biases"])
    spiralLayer2.setWeights(savedModel["layer2"]["weights"])
    spiralLayer2.setBiases(savedModel["layer2"]["biases"])
    print("Successfully loaded 'trainedModel.json'!")

except FileNotFoundError:
    print("Error: 'trainedModel.json' not found. Training from scratch.")

# softmaxCrossEntropy = Loss.SoftmaxCrossEntropy()
softmaxCrossEntropy = Loss.SoftmaxCrossEntropy_batch()
spiralOptimizer = Optimizer.Optimizer_SGD(learning_rate=0.01)
accuracyCalculator = Accuracy.Accuracy_hard()

for epoch in range(1_000):
    numSamples = len(X)

    for i in range(0, numSamples, BATCH_SIZE):
        X_batch = X[i : i + BATCH_SIZE]
        y_batch = y[i : i + BATCH_SIZE]

        ### Forwards pass
        ### layer1 -> layer2 -> layer3 -> softmax_error
        layer1Output = spiralLayer1.forward(X_batch)                              #< returns a list of lists of outputs
        layer2Output = spiralLayer2.forward(layer1Output)                         #< returns a list of lists of outputs
        # print(layer2Output)
        error = softmaxCrossEntropy.forward(layer2Output, y_batch)                #< returns mean error
        accuracy = accuracyCalculator.epoch(layer2Output, y_batch)                #< returns accuracy of model %right

        ### Backwards pass
        ### error_softmax -> layer2 ->  layer1 
        d_crossEntropy = softmaxCrossEntropy.backward()
        d_layer2 = spiralLayer2.backward(d_crossEntropy)
        # print([Mathlib.mean(d) for d in d_layer2])
        d_layer1 = spiralLayer1.backward(d_layer2)
        # print([Mathlib.mean(d) for d in d_layer1])

        spiralOptimizer.update(spiralLayer2)
        spiralOptimizer.update(spiralLayer1)

    if epoch%10 == 0:
        print(f"Epoch {epoch}: Error: {error}, accuracy: {accuracy}")  #< Normal to bounce around at the start
        modelState = {
            "layer1": {
                "weights": spiralLayer1.getWeights(),
                "biases": spiralLayer1.getBiases()
            },
            "layer2": {
                "weights": spiralLayer2.getWeights(),
                "biases": spiralLayer2.getBiases()
            }
        }
        with open("trainedModel.json", "w") as f:
            json.dump(modelState, f, indent=4)

end = time.time()
print(f"Training took: {end-start} seconds")


Converting data to hilbert space
Done/1000
Error: 'trainedModel.json' not found. Training from scratch.
Epoch 0: Error: 0.6971276506890968, accuracy: 0.527
Epoch 10: Error: 0.6598023840945021, accuracy: 0.5774545454545454
Epoch 20: Error: 0.6286918699234699, accuracy: 0.608
Epoch 30: Error: 0.5987478963414038, accuracy: 0.6318709677419355
Epoch 40: Error: 0.5714136948481052, accuracy: 0.6515609756097561
Epoch 50: Error: 0.5418469890648122, accuracy: 0.6682549019607843
Epoch 60: Error: 0.5175948952231373, accuracy: 0.682311475409836
Epoch 70: Error: 0.49471004093047793, accuracy: 0.6947042253521126
Epoch 80: Error: 0.47418168455586934, accuracy: 0.7055802469135802
Epoch 90: Error: 0.4791154200508596, accuracy: 0.7149230769230769
Epoch 100: Error: 0.40683205518971116, accuracy: 0.7241683168316831
Epoch 110: Error: 0.3893268037675091, accuracy: 0.7328918918918919
Epoch 120: Error: 0.37286226225566166, accuracy: 0.7407190082644628
Epoch 130: Error: 0.36010251854390013, accuracy: 0.74864885

### The raw Python model takes about 45 minutes to train to 90% and slightly less than 90 minutes to fully train.

### The C++ model takes about 2 minutes to train to 90% accuracy and slightly more than 4 minutes to fully train Thats 21x the speed!

In [7]:
import Demo
import tkinter as tk
root = tk.Tk()
app = Demo.DrawingApp(root)
root.mainloop()

[Model Inference] SQUARE! (Confidence: 99.6%)
[Model Inference] NOT A SQUARE! (Confidence: 95.4%)
[Model Inference] SQUARE! (Confidence: 68.2%)
[Model Inference] NOT A SQUARE! (Confidence: 99.3%)
[Model Inference] NOT A SQUARE! (Confidence: 59.5%)
